# Lesson 2.1 · What a tool call is: the model asks, your code acts

**Goal:** see, with real replies, that a model never runs a tool. It writes a request; your code decides, runs it and sends the result back.

**What you will be able to do**
- Read a tool call: its `id`, its `name`, and its `arguments` (a JSON **string**).
- Parse the arguments, run the function, and append a `tool` message with the same `tool_call_id`.
- Count your own tool runs, so you can tell a checked answer from an invented one.
- Spot the four beginner traps before they bite.

> Cells marked **calls the API** talk to Groq (free key, lesson 0.2). Their replies are saved in `./cache`, so the notebook re-runs for free and works without a key. There are 4 such cells, plus 1 optional one in exercise 1.

In [1]:
# Setup: run this cell first.
# It finds the course's nightwatch/ folder (the Brightcart simulator), loads GROQ_API_KEY from .env if there is one,
# and builds `chat(...)`: a client call with a record/replay cache in ./cache, so re-runs cost nothing.
import hashlib, json, os, sys
from pathlib import Path

HERE = Path.cwd()
NW = next((p / "nightwatch" for p in [HERE, *HERE.parents] if (p / "nightwatch" / "brightcart").is_dir()), None)
assert NW is not None, "Can't find nightwatch/. Open this notebook from inside the course folder (lessons/<lesson>/notebook)."
sys.path.insert(0, str(NW))
from brightcart.incidents import fresh, generate
from brightcart.sim import World

try:
    from dotenv import load_dotenv
    load_dotenv(NW.parent / ".env")
except ImportError:
    pass
KEY = os.environ.get("GROQ_API_KEY")

import httpx
from openai import BadRequestError, OpenAI
from openai.types.chat import ChatCompletion

client = OpenAI(api_key=KEY or "missing", base_url="https://api.groq.com/openai/v1", max_retries=8)
MODEL = "openai/gpt-oss-20b"
CACHE = HERE / "cache"


def chat(sample=0, **body):
    """One model call with record/replay: a cached reply is replayed for free, a new one calls Groq once and is saved.
    A 400 (malformed tool call) is cached too and raised again as BadRequestError, like the real client does."""
    h = hashlib.sha256((json.dumps(body, sort_keys=True, default=str) + f"#{sample}").encode()).hexdigest()[:20]
    f = CACHE / f"{h}.json"
    if f.exists():
        saved = json.loads(f.read_text(encoding="utf-8"))
        if "error_400" in saved:
            raise BadRequestError(saved["error_400"], body=None,
                                  response=httpx.Response(400, request=httpx.Request("POST", str(client.base_url))))
        saved.pop("service_tier", None)            # Groq sends "on_demand", which the SDK type does not list
        return ChatCompletion.model_validate(saved)
    if not KEY:
        raise RuntimeError("This reply is not in ./cache and GROQ_API_KEY is not set. Get the free key (lesson 0.2), "
                           "put GROQ_API_KEY=... in the course's .env, and re-run the setup cell.")
    CACHE.mkdir(exist_ok=True)
    try:
        r = client.chat.completions.create(**body)
    except BadRequestError as e:
        f.write_text(json.dumps({"error_400": str(e)}), encoding="utf-8")
        raise
    f.write_text(r.model_dump_json(), encoding="utf-8")
    return r


def check(n, test):
    """Run an exercise check; print a clear line instead of stopping the notebook."""
    try:
        test()
        print(f"✓ Exercise {n} passed")
    except Exception as e:
        print(f"✗ Exercise {n} not yet: {type(e).__name__}: {str(e) or 'an assert failed, see the check cell'}")


print("nightwatch found:", NW.name, "| GROQ_API_KEY:", "set" if KEY else
      "NOT set -> cells marked 'calls the API' replay from ./cache; a new question needs the free key (lesson 0.2)")

nightwatch found: nightwatch | GROQ_API_KEY: set


## 1. Recall: structured output (lesson 1.3)

A model reply was safe to use in code when it was JSON with a fixed shape that our code **parsed and checked**. A tool call is the same idea, addressed to your code and asking it to do something.

In [2]:
reply = '{"service": "payments", "severity": "high"}'   # text from a model
data = json.loads(reply)                                 # text -> dict, then check it
print(type(reply).__name__, "->", type(data).__name__, "| service:", data["service"])

str -> dict | service: payments


## 2. Key words

| Word | Meaning |
|---|---|
| tool | a plain Python function your code is willing to run for the model |
| tool schema | the JSON card that describes a tool to the model: name, description, parameters |
| tool call | the model's request to run one tool, as a small piece of JSON |
| arguments | the values the model fills in (which service, how many lines), sent as a JSON **string** |
| tool result | what your function returned, sent back to the model |
| tool call ID | the label that ties each result to the call it answers |
| `finish_reason == "tool_calls"` | the model stopped to ask for a tool, not to answer |

## 3. The idea: the model proposes, your code acts

The model reads text and writes text. The most it can do is **propose** a tool call. Think of a restaurant: the model is the waiter who writes the order (`read_logs`, `checkout`, 10 lines); your code is the kitchen, the only one that touches the stove (logs, database, servers). The kitchen can cook the order or refuse it.

After one round trip the message list holds four messages:

| # | role | content |
|---|---|---|
| 1 | system | "You are Nightwatch, the on-call assistant for the shop Brightcart." |
| 2 | user | "Checkout errors spiked. What do the checkout logs say? ..." |
| 3 | assistant | no text, one tool call: `id`, `read_logs`, `'{"n":10,"service":"checkout"}'` |
| 4 | tool | the 10 log lines as JSON text, with `tool_call_id` = the id from message 3 |

### Hand trace (the video's run)

| step | what happens | `finish_reason` | `calls_run` |
|---|---|---|---|
| call 1, no tools | model answers "502 Bad Gateway from the payment gateway" | stop | 0 |
| call 2, with `read_logs` | content `None`, one tool call, arguments `'{"n":10,"service":"checkout"}'` (a `str`) | tool_calls | 0 |
| your code: `json.loads` | `{'n': 10, 'service': 'checkout'}` | | 0 |
| your code: `read_logs(**args)` | 10 lines, 4 ERRORs at 04:13: `upstream payments returned 500` | | **1** |
| append tool message | roles: system, user, assistant, tool | | 1 |
| call 3, all 4 messages | answer from the logs: payments returned 500 | stop | 1 |

In the video's run the last call used **382 prompt + 195 completion** tokens.

## 4. Write it yourself

The Brightcart simulator, incident `upstream_deploy-0` (checkout errors; the real cause is a payments deploy). `calls_run` counts how often **our code** really reads the logs.

In [3]:
inc = next(x for x in generate() if x.id == "upstream_deploy-0")
world = World(fresh(inc))
calls_run = 0


def read_logs(service, n=5):
    global calls_run
    calls_run += 1
    return world.logs(service, n)


TOOLS = [{"type": "function", "function": {
    "name": "read_logs",
    "description": "Return the last n log lines of one Brightcart service.",
    "parameters": {"type": "object",
                   "properties": {"service": {"type": "string", "description": "e.g. checkout, payments"},
                                  "n": {"type": "integer", "description": "how many lines"}},
                   "required": ["service"]}}}]
SYSTEM = "You are Nightwatch, the on-call assistant for the shop Brightcart."
ASK = "Checkout errors spiked. What do the checkout logs say? Answer in one sentence."
print("services:", list(world.inc.services))

services: ['web', 'checkout', 'payments', 'inventory', 'search', 'email-worker']


### 4.1 No tools: ask the model to read the logs

In [4]:
# calls the API (replays from ./cache when the reply is saved)
r = chat(model=MODEL, temperature=0,
         messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": ASK}])
print("reply:", r.choices[0].message.content)
print("times our code ran read_logs:", calls_run)

reply: The checkout logs show a sudden spike in 502 Bad Gateway errors from the payment gateway, indicating a temporary outage that caused the checkout failures.
times our code ran read_logs: 0


A confident answer, and our code read the logs **zero** times. (The video's run said "502 Bad Gateway errors from the payment gateway"; your wording may differ, the zero will not.)

### 4.2 With a tool: the model asks

In [5]:
# calls the API (replays from ./cache when the reply is saved)
msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ASK}]
r = chat(model=MODEL, temperature=0, messages=msgs, tools=TOOLS)
m = r.choices[0].message
print("finish_reason:", r.choices[0].finish_reason)
print("content:", repr(m.content))
for c in m.tool_calls:
    print("tool_call:", json.dumps({"id": c.id, "name": c.function.name, "arguments": c.function.arguments}))
print("type of arguments:", type(m.tool_calls[0].function.arguments).__name__)
print("times our code ran read_logs so far:", calls_run)

finish_reason: tool_calls
content: None
tool_call: {"id": "fc_b402c379-e7b4-4909-8eb8-acfdccd844ee", "name": "read_logs", "arguments": "{\"n\":10,\"service\":\"checkout\"}"}
type of arguments: str
times our code ran read_logs so far: 0


### 4.3 Your code acts: parse, run, append

Three moves: put the model's own message (with its tool calls) into the history, then for each call parse the arguments string, run the function with `**args`, and append a `tool` message with the **same id**.

In [6]:
msgs.append({"role": "assistant", "content": m.content or "",
             "tool_calls": [c.model_dump() for c in m.tool_calls]})
for c in m.tool_calls:
    args = json.loads(c.function.arguments)          # str -> dict
    result = read_logs(**args)                        # ** spreads the dict into keyword arguments
    print(f"ran read_logs({args}) ->")
    for line in result:
        print("   ", line)
    msgs.append({"role": "tool", "tool_call_id": c.id, "content": json.dumps(result)})
print("times our code ran read_logs:", calls_run)
print("messages now:", [x["role"] for x in msgs])

ran read_logs({'n': 10, 'service': 'checkout'}) ->
    04:05 INFO checkout handled request id=85086 in 179ms
    04:09 INFO checkout handled request id=93973 in 121ms
    04:10 WARN checkout slow GC pause 120ms
    04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge
    04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge
    04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge
    04:13 ERROR checkout upstream payments returned 500 for POST /payments/v2/charge
    04:36 INFO checkout handled request id=57805 in 137ms
    04:49 INFO checkout handled request id=31305 in 53ms
    04:59 INFO checkout handled request id=58607 in 149ms
times our code ran read_logs: 1
messages now: ['system', 'user', 'assistant', 'tool']


In [7]:
# calls the API (replays from ./cache when the reply is saved)
r = chat(model=MODEL, temperature=0, messages=msgs, tools=TOOLS)
print("finish_reason:", r.choices[0].finish_reason)
print("final answer:", r.choices[0].message.content)
print("last call re-sent", len(msgs), "messages | usage:", r.usage.prompt_tokens, "prompt +",
      r.usage.completion_tokens, "completion")

finish_reason: stop
final answer: The checkout logs show a spike of errors with the upstream payments service returning 500 for POST /payments/v2/charge, along with a warning about a slow GC pause.
last call re-sent 4 messages | usage: 382 prompt + 198 completion


Now the answer comes from the logs: payments returned 500 for `POST /payments/v2/charge`. The video's last call used 382 prompt + 195 completion tokens; the tool result rides along in every later call, so keep results short (lesson 2.2).

## 5. SDK objects vs plain dicts + check

The OpenAI SDK gives typed objects (`c.function.arguments`). What goes back into `messages` is plain dicts. Two checks: `model_dump()` keeps the id and the arguments string unchanged, and Nightwatch's own tool runner (`agent_v0.run_tool`) returns exactly the lines our hand-written `read_logs` returned.

In [8]:
import openai
import agent_v0

c = m.tool_calls[0]
d = c.model_dump()
print("openai", openai.__version__)
print("dump keeps id:", d["id"] == c.id, "| keeps arguments string:", d["function"]["arguments"] == c.function.arguments)
args = json.loads(c.function.arguments)
before = calls_run
ours = read_logs(**args)
theirs = agent_v0.run_tool(world, "read_logs", args)
print("same lines as Nightwatch's run_tool:", ours == theirs, "| our counter moved by", calls_run - before)

openai 3.51.0
dump keeps id: True | keeps arguments string: True
same lines as Nightwatch's run_tool: True | our counter moved by 1


## 6. Where this shows up in agents

Every action Nightwatch takes in this course is this same exchange. Reads (`read_logs`) and writes (`rollback_deploy`) alike: the model can only **ask**. Your code can run it, ask a human first (chapter 9), or refuse, and the refusal goes back as the tool result. Checking arguments comes in lesson 2.4.

In [9]:
names = [t["function"]["name"] for t in agent_v0.TOOLS]
print(len(names), "Nightwatch tools:", ", ".join(names))
print("writes the model can only ask for:", sorted(agent_v0.WRITE_TOOLS))

17 Nightwatch tools: service_status, read_logs, recent_deploys, list_flags, config_changes, queue_depth, query_db, read_runbook, check_health, restart_service, rollback_deploy, scale_service, set_flag, set_config, rotate_secret, clear_cache, finish
writes the model can only ask for: ['clear_cache', 'restart_service', 'rollback_deploy', 'rotate_secret', 'scale_service', 'set_config', 'set_flag']


## 7. Traps (each one runs and shows its error)

**Trap 1: treating the arguments as a dict.** They arrive as a string.

In [10]:
raw = m.tool_calls[0].function.arguments
try:
    raw["service"]
except TypeError as e:
    print("TypeError:", e, "-> parse first:", json.loads(raw)["service"])

TypeError: string indices must be integers, not 'str' -> parse first: checkout


**Trap 2: sending the tool result without its `tool_call_id`.** Nothing ties the result to the request it answers. (This cell calls the API once to show the server's reply.)

In [11]:
# calls the API (replays from ./cache when the reply is saved)
broken = [dict(x) for x in msgs]
del broken[-1]["tool_call_id"]
try:
    chat(model=MODEL, temperature=0, messages=broken, tools=TOOLS)
    print("accepted (this server did not complain)")
except BadRequestError as e:
    print("400:", str(e)[:220])

400: Error code: 400 - 'messages.3' : for 'role:tool' the following must be satisfied[('messages.3.tool_call_id' : property 'tool_call_id' is missing)]


**Trap 3: believing the tool ran because the model said so.** Only your code runs tools, so count your own calls.

In [12]:
calls_before = calls_run
r_no_tool = chat(model=MODEL, temperature=0,                       # replayed from the cache (cell 4.1)
                 messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": ASK}])
print("model says:", r_no_tool.choices[0].message.content[:90], "...")
print("our code ran read_logs during that answer:", calls_run - calls_before, "times")

model says: The checkout logs show a sudden spike in 502 Bad Gateway errors from the payment gateway,  ...
our code ran read_logs during that answer: 0 times


**Trap 4: running whatever arguments arrive.** A wrong name or an unknown tool must be checked before you act.

In [13]:
TOOL_FUNCS = {"read_logs": read_logs}
for name, arguments in [("read_logs", '{"service": "payment"}'), ("drop_database", "{}")]:
    try:
        TOOL_FUNCS[name](**json.loads(arguments))
    except KeyError as e:
        print(f"{name}({arguments}) -> KeyError: {e}")

read_logs({"service": "payment"}) -> KeyError: "unknown service 'payment'"
drop_database({}) -> KeyError: 'drop_database'


## 8. Exercises

Check cells never call the API: they feed your code hand-written tool calls in the same shape as `c.model_dump()`.

### Exercise 1 (the video's exercise)
Ask about the **payments** service instead. Write `ASK2` (a question about the payments logs) and `handle(tool_calls)`: it takes a list of tool calls as dicts (`{"id", "function": {"name", "arguments"}}`), runs `read_logs` for each, and returns the list of `tool` messages. Print how many times `read_logs` ran before and after. The optional cell below runs it against the model.

In [14]:
ASK2 = ""   # YOUR CODE HERE


def handle(tool_calls):
    # YOUR CODE HERE: parse each call's arguments, run read_logs, return one tool message per call
    return []

In [15]:
# calls the API (replays from ./cache when the reply is saved)
# optional: the real exchange for your ASK2 (skips if ASK2 is still empty)
if ASK2:
    m2 = chat(model=MODEL, temperature=0, tools=TOOLS,
              messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": ASK2}]).choices[0].message
    for c in m2.tool_calls or []:
        print("tool_call:", c.function.name, c.function.arguments)
    before = calls_run
    out = handle([c.model_dump() for c in m2.tool_calls or []])
    print("read_logs ran", before, "->", calls_run, "| tool messages:", len(out))

In [16]:
def _t1():
    assert "payments" in ASK2.lower(), "ASK2 should ask about the payments service"
    fake = [{"id": "call_pay1", "type": "function",
             "function": {"name": "read_logs", "arguments": '{"service": "payments", "n": 3}'}}]
    before = calls_run
    out = handle(fake)
    assert calls_run - before == 1, f"read_logs should run once, ran {calls_run - before} times"
    assert len(out) == 1 and out[0]["role"] == "tool", "return one message with role 'tool'"
    assert out[0]["tool_call_id"] == "call_pay1", "copy the call's id into tool_call_id"
    assert json.loads(out[0]["content"]) == world.logs("payments", 3), "content = the lines as JSON text"
check(1, _t1)

✗ Exercise 1 not yet: AssertionError: ASK2 should ask about the payments service


### Exercise 2: safe argument parsing
Write `parse_args(arguments)`: return a dict from the JSON string; treat `""` or `None` as `{}`; raise `ValueError` if the JSON is not an object (for example `"[1, 2]"`).

In [17]:
def parse_args(arguments):
    # YOUR CODE HERE
    pass

In [18]:
def _t2():
    assert parse_args('{"n":10,"service":"checkout"}') == {"n": 10, "service": "checkout"}
    assert parse_args("") == {} and parse_args(None) == {}, "empty arguments mean no arguments"
    try:
        parse_args("[1, 2]")
    except ValueError:
        pass
    else:
        raise AssertionError("a JSON list is not an arguments object: raise ValueError")
check(2, _t2)

✗ Exercise 2 not yet: AssertionError: an assert failed, see the check cell


### Exercise 3: the kitchen can refuse
Write `run_if_allowed(name, args)`: run `read_logs(**args)` only when `name == "read_logs"` and `args["service"]` is one of `world.inc.services`; otherwise return a string that starts with `"REFUSED:"` and **do not** run anything.

In [19]:
def run_if_allowed(name, args):
    # YOUR CODE HERE
    pass

In [20]:
def _t3():
    before = calls_run
    assert run_if_allowed("read_logs", {"service": "checkout", "n": 2}) == world.logs("checkout", 2)
    assert str(run_if_allowed("read_logs", {"service": "payment"})).startswith("REFUSED:"), "unknown service"
    assert str(run_if_allowed("rollback_deploy", {"service": "payments"})).startswith("REFUSED:"), "unknown tool"
    assert calls_run - before == 1, "only the allowed call may run"
check(3, _t3)

✗ Exercise 3 not yet: AssertionError: an assert failed, see the check cell


### Exercise 4: check the history before you send it
Write `history_problems(messages)`: return a list of problems (strings). A `tool` message without `tool_call_id` is a problem, and so is a `tool_call_id` that no earlier assistant tool call used. A clean history returns `[]`.

In [21]:
def history_problems(messages):
    # YOUR CODE HERE
    return None

In [22]:
def _t4():
    assert history_problems(msgs) == [], "the history from section 4 is clean"
    no_id = [dict(x) for x in msgs]; del no_id[-1]["tool_call_id"]
    assert len(history_problems(no_id)) == 1, "a tool message without tool_call_id"
    wrong = [dict(x) for x in msgs]; wrong[-1]["tool_call_id"] = "call_nobody"
    assert len(history_problems(wrong)) == 1, "a tool_call_id that no call used"
check(4, _t4)

✗ Exercise 4 not yet: AssertionError: the history from section 4 is clean


## Solutions (try first!)

In [23]:
ASK2 = "Payments looks unhappy. What do the payments logs say? Answer in one sentence."


def handle(tool_calls):
    out = []
    for c in tool_calls:
        args = json.loads(c["function"]["arguments"] or "{}")
        result = read_logs(**args)
        out.append({"role": "tool", "tool_call_id": c["id"], "content": json.dumps(result)})
    return out

In [24]:
def parse_args(arguments):
    if not arguments:
        return {}
    data = json.loads(arguments)
    if not isinstance(data, dict):
        raise ValueError(f"arguments must be a JSON object, got {type(data).__name__}")
    return data

In [25]:
def run_if_allowed(name, args):
    if name != "read_logs":
        return f"REFUSED: unknown tool {name!r}"
    if args.get("service") not in world.inc.services:
        return f"REFUSED: unknown service {args.get('service')!r}; known: {', '.join(world.inc.services)}"
    return read_logs(**args)

In [26]:
def history_problems(messages):
    problems, seen = [], set()
    for i, x in enumerate(messages):
        if x["role"] == "assistant":
            seen |= {c["id"] for c in x.get("tool_calls") or []}
        elif x["role"] == "tool":
            if "tool_call_id" not in x:
                problems.append(f"message {i}: tool message without tool_call_id")
            elif x["tool_call_id"] not in seen:
                problems.append(f"message {i}: tool_call_id {x['tool_call_id']!r} answers no call")
    return problems